En este cuaderno vamos usar 3 modelos ViT (Vision Transformers). Los elegidos son:
- *google/vit-base-patch16-224-in21k*
- *microsoft/swin-tiny-patch4-window7-224*
- *facebook/convnext-tiny-224*

Los entrenaremos con diferentes datasets creados en el otro cuaderno. Los datasets han sido creados extrayendo los frames a partir del vídeo, tenememos datasets con 2, 4, 6, 8 y 10 frames.

Al que mejor resultado nos dé le haremos un preprocesamiento a los frames y optimización de hiperparámetros.

In [91]:
!pip install evaluate decord

In [92]:
import pandas as pd
import numpy as np
import torch
import os
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix
from datasets import Dataset, Image, Features, Value
from transformers import (
    ViTImageProcessor,
    ViTForImageClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    AutoImageProcessor,
    AutoModelForImageClassification
)
import evaluate
from PIL import Image
from tqdm import tqdm

from torchvision.transforms import (
    Compose,
    RandomHorizontalFlip,
    ColorJitter,
    RandomRotation
)

In [93]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 1. Entrenamiento Train/Valid

## 1.1. Parametros de los experimentos

A modificar según se necesite

In [94]:
NUM_FOTOGRAMAS_POR_VIDEO = 2
#NUM_FOTOGRAMAS_POR_VIDEO = 4
#NUM_FOTOGRAMAS_POR_VIDEO = 6
#NUM_FOTOGRAMAS_POR_VIDEO = 8
#NUM_FOTOGRAMAS_POR_VIDEO = 10

#MODELO_ELEGIDO = "swin"
#MODELO_ELEGIDO = "vit"
MODELO_ELEGIDO = "convnext"

In [95]:
# Diccionario de modelos
MODEL_ZOO = {
    "vit": "google/vit-base-patch16-224-in21k",
    "swin": "microsoft/swin-tiny-patch4-window7-224",
    "convnext": "facebook/convnext-tiny-224"
}

MODEL_CHECKPOINT = MODEL_ZOO[MODELO_ELEGIDO]

In [96]:
# Construcción dinámica de las rutas
CSV_IMAGENES = f"/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Frames/Frames-{NUM_FOTOGRAMAS_POR_VIDEO}/dataset_imagenes_train_{NUM_FOTOGRAMAS_POR_VIDEO}_Frames.csv"
CSV_TRAIN_MASTER_TEXT = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/EXIST2025_training_3_1_master.csv"
#OUTPUT_DIR = f"/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Vision/{MODELO_ELEGIDO.upper()}_Frames_{NUM_FOTOGRAMAS_POR_VIDEO}"

# Nueva ruta para no perder el modelo toqueteando nuevas cosas (hiperparametros y data augmentation)
OUTPUT_DIR = f"/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Vision/{MODELO_ELEGIDO.upper()}_Frames_{NUM_FOTOGRAMAS_POR_VIDEO}_experimentacion"

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 1.2. Carga de Datos y alineación estática

In [97]:
print(f"Lanzando experimento con {MODELO_ELEGIDO.upper()} y {NUM_FOTOGRAMAS_POR_VIDEO} Frames/Vídeo")
print("Cargando el dataset de imágenes y la plantilla estática...")

df_imagenes = pd.read_csv(CSV_IMAGENES)
df_train_text = pd.read_csv(CSV_TRAIN_MASTER_TEXT)

# Nos aseguramos de entrenar SOLO con los vídeos del Train Master estático
train_master_ids = df_train_text['id_EXIST'].unique()
df_train_master = df_imagenes[df_imagenes['id_EXIST'].isin(train_master_ids)].copy()

Lanzando experimento con CONVNEXT y 2 Frames/Vídeo
Cargando el dataset de imágenes y la plantilla estática...


## 1.3. División dinámica Train/Valid con prevención de leakage

In [98]:
# GroupShuffleSplit asegura que los frames del mismo vídeo vayan al mismo conjunto
gss_train_val = GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=42)

train_idx, val_idx = next(gss_train_val.split(df_train_master, groups=df_train_master['id_EXIST']))

train_df = df_train_master.iloc[train_idx].copy()
val_df = df_train_master.iloc[val_idx].copy()

print("\n--- DISTRIBUCIÓN DEL DATASET DE IMÁGENES ---")
print(f"Vídeos en Train: {train_df['id_EXIST'].nunique()} (Total: {len(train_df)} fotogramas)")
print(f"Vídeos en Valid: {val_df['id_EXIST'].nunique()} (Total: {len(val_df)} fotogramas)")


--- DISTRIBUCIÓN DEL DATASET DE IMÁGENES ---
Vídeos en Train: 1805 (Total: 3609 fotogramas)
Vídeos en Valid: 201 (Total: 402 fotogramas)


## 1.4. Preparación dataset Hugging Face y procesador universal

In [99]:
from datasets import Image as DatasetsImage

def crear_dataset(dataframe):
    dataset = Dataset.from_pandas(dataframe)
    dataset = dataset.cast_column("path_imagen", DatasetsImage(decode=False))
    return dataset.rename_column("path_imagen", "image")

train_dataset = crear_dataset(train_df)
valid_dataset = crear_dataset(val_df)

In [100]:
# AutoImageProcessor detecta automáticamente qué preprocesado necesita (ViT, Swin o ConvNeXt)
processor = AutoImageProcessor.from_pretrained(MODEL_CHECKPOINT)

"""def process_images(batch):
    imagenes_validas = []

    for img_dict in batch["image"]:
        try:
            # Hugging Face devuelve un diccionario {'path': ..., 'bytes': ...} cuando decode=False
            ruta = img_dict['path']
            # Intentamos abrir y decodificar manualmente
            img = Image.open(ruta).convert("RGB")
            imagenes_validas.append(img)
        except Exception as e:
            # 🛡️ EL SALVAVIDAS: Si el archivo está corrupto, inyectamos una imagen negra
            print(f"⚠️ Imagen corrupta detectada y reemplazada por negro: {ruta}")
            imagen_negra = Image.new("RGB", (224, 224), (0, 0, 0))
            imagenes_validas.append(imagen_negra)

    # Pasamos las imágenes (ahora 100% seguras) al procesador matemático
    inputs = processor(imagenes_validas, return_tensors="pt")
    inputs["labels"] = batch["label"]
    return inputs"""

# DATA AURGMENTATION (Descomentar en caso de necesitarlo)

_train_transforms = Compose([
    RandomHorizontalFlip(p=0.5),
    RandomRotation(degrees=10),
    ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
])

def process_images_train(batch):
    imagenes_transformadas = []

    for item, etiqueta in zip(batch["image"], batch["label"]):
        try:
            if isinstance(item, Image.Image):
                img = item.convert("RGB")
            elif isinstance(item, dict) and 'path' in item:
                img = Image.open(item['path']).convert("RGB")
            elif isinstance(item, str):
                img = Image.open(item).convert("RGB")
            else:
                raise ValueError("Formato desconocido")

            # 🎯 LÓGICA ESTOCÁSTICA 🎯
            # Si es clase 1 o 0 (comentar segun conveniencia) Y además ganamos el "sorteo" del 10% de probabilidad
            if etiqueta == 0: #and random.random() < 0.10:
                img_final = _train_transforms(img)
            # En el 90% de los casos de clase 1, o si es clase 0, la dejamos intacta
            else:
                img_final = img

            # ¡APLICAMOS DATA AUGMENTATION SOLO EN TRAIN! (Para todas las imágenes sin importar las etiquetas)
            """img_aug = _train_transforms(img)
            imagenes_transformadas.append(img_aug)"""

            imagenes_transformadas.append(img_final)

        except Exception:
            imagenes_transformadas.append(Image.new("RGB", (224, 224), (0, 0, 0)))

    inputs = processor(imagenes_transformadas, return_tensors="pt")
    inputs["labels"] = batch["label"]
    return inputs

def process_images_val(batch):
    imagenes_validas = []
    for item in batch["image"]:
        try:
            if isinstance(item, Image.Image):
                img = item.convert("RGB")
            elif isinstance(item, dict) and 'path' in item:
                img = Image.open(item['path']).convert("RGB")
            elif isinstance(item, str):
                img = Image.open(item).convert("RGB")
            else:
                raise ValueError("Formato desconocido")
            imagenes_validas.append(img)
        except Exception:
            imagenes_validas.append(Image.new("RGB", (224, 224), (0, 0, 0)))

    inputs = processor(imagenes_validas, return_tensors="pt")
    inputs["labels"] = batch["label"]
    return inputs

train_dataset.set_transform(process_images_train)
valid_dataset.set_transform(process_images_val)

## 1.5. Carga del modelo universal

In [101]:
id2label = {0: "No misógino", 1: "Misógino"}
label2id = {"No misógino": 0, "Misógino": 1}

In [102]:
model = AutoModelForImageClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/182 [00:00<?, ?it/s]

[transformers] ConvNextForImageClassification LOAD REPORT from: facebook/convnext-tiny-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([2])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([2, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


In [103]:
def collate_fn(batch):
    return {
        'pixel_values': torch.stack([x['pixel_values'] for x in batch]),
        'labels': torch.tensor([x['labels'] for x in batch])
    }

## 1.6. Métricas y configuración del trainer

In [104]:
f1_metric = evaluate.load("f1")
accuracy_metric = evaluate.load("accuracy")

In [105]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    return {"f1": f1, "accuracy": acc}

In [106]:
# Parámetros por defecto
LR_MANUAL = 5e-5
WD_MANUAL = 0.01
WARMUP_RATIO = 0.0
EPOCHS_MANUAL = 6

# Hiperparámetros 1
"""LR_MANUAL = 1e-4
WD_MANUAL = 0.05
WARMUP_RATIO = 0.1
EPOCHS_MANUAL = 8"""

# Hiperparámetros 2
"""LR_MANUAL = 2e-4
WD_MANUAL = 0.05
WARMUP_RATIO = 0.1
EPOCHS_MANUAL = 6"""

#Slow cooker (mayor regularización y paciencia)
"""LR_MANUAL = 5e-5
WD_MANUAL = 0.1
WARMUP_RATIO = 0.1
EPOCHS_MANUAL = 8"""

#Batch acumulado (simular memoria más grande)
"""LR_MANUAL = 5e-5
WD_MANUAL = 0.01
WARMUP_RATIO = 0.0
EPOCHS_MANUAL = 8
#gradient_accumulation_steps = 2 --> esto se hace en el TrainingArguments"""

#LR Scheduler Agresivo (El sprint final)
"""WD_MANUAL = 0.05
LR_MANUAL = 5e-5
#lr_scheduler_type = "cosine" --> esto se hace en el TrainingArguments
EPOCHS_MANUAL = 6
WARMUP_RATIO = 0.1"""

#Slow cooker extendido (subimos el número de épocas a 10)
"""LR_MANUAL = 5e-5
WD_MANUAL = 0.1
WARMUP_RATIO = 0.1
EPOCHS_MANUAL = 10
# Comentamos el early_stopping para desactivarlo"""

'LR_MANUAL = 5e-5\nWD_MANUAL = 0.1\nWARMUP_RATIO = 0.1\nEPOCHS_MANUAL = 10\n# Comentamos el early_stopping para desactivarlo'

In [107]:
# 1. Calculamos los pasos totales del entrenamiento
total_train_steps = (len(train_dataset) // 16) * EPOCHS_MANUAL

# 2. Obtenemos el número entero de pasos para el warmup
warmup_steps_calculados = int(total_train_steps * WARMUP_RATIO)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    #learning_rate=5e-5,
    learning_rate = LR_MANUAL,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1, # --> el valor debe ser 1 a no ser que esté ejecutando la combinación batch_acumulado
    #num_train_epochs=5,
    num_train_epochs = EPOCHS_MANUAL,
    #weight_decay=0.01,
    weight_decay = WD_MANUAL,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_strategy="steps",
    logging_steps=50,
    fp16=True,
    report_to="none",
    warmup_steps=warmup_steps_calculados,
    #lr_scheduler_type = "cosine", # --> Correspondiente a la estrategia de lr_scheduler agresivo
)

In [108]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=collate_fn,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics,
    #callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

## 1.7. Entrenamiento

In [109]:
print(f"🚀 Iniciando entrenamiento visual con {MODELO_ELEGIDO.upper()}...")
trainer.train()

print("💾 Guardando modelo visual...")
trainer.save_model(os.path.join(OUTPUT_DIR, "modelo_final"))
processor.save_pretrained(os.path.join(OUTPUT_DIR, "modelo_final"))
print("✅ ¡Entrenamiento completado!")

🚀 Iniciando entrenamiento visual con CONVNEXT...


Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.375321,0.950087,0.440821,0.497512
2,0.181678,1.263450,0.434701,0.497512
3,0.169297,1.471486,0.465693,0.504975
4,0.112909,1.594266,0.493503,0.522388
5,0.092447,1.608058,0.542773,0.554726
6,0.064608,1.917653,0.458423,0.497512


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

💾 Guardando modelo visual...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ ¡Entrenamiento completado!


# 2. Validación contra fichero de test estático

In [110]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 2.1. Parámetros del experimento

In [111]:
"""NUM_FOTOGRAMAS_POR_VIDEO = 2
#NUM_FOTOGRAMAS_POR_VIDEO = 4
#NUM_FOTOGRAMAS_POR_VIDEO = 6
#NUM_FOTOGRAMAS_POR_VIDEO = 8
#NUM_FOTOGRAMAS_POR_VIDEO = 10

MODELO_ELEGIDO = "swin"
#MODELO_ELEGIDO = "vit"
#MODELO_ELEGIDO = "convnext" """

'NUM_FOTOGRAMAS_POR_VIDEO = 2\n#NUM_FOTOGRAMAS_POR_VIDEO = 4\n#NUM_FOTOGRAMAS_POR_VIDEO = 6\n#NUM_FOTOGRAMAS_POR_VIDEO = 8\n#NUM_FOTOGRAMAS_POR_VIDEO = 10\n\nMODELO_ELEGIDO = "swin"\n#MODELO_ELEGIDO = "vit"\n#MODELO_ELEGIDO = "convnext" '

In [112]:
# Rutas dinámicas
CSV_IMAGENES_TEST = f"/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Frames/Frames-{NUM_FOTOGRAMAS_POR_VIDEO}/dataset_imagenes_test_{NUM_FOTOGRAMAS_POR_VIDEO}_Frames.csv"
#MODEL_DIR = f"/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Vision/{MODELO_ELEGIDO.upper()}_Frames_{NUM_FOTOGRAMAS_POR_VIDEO}_experimentacion/modelo_final"
#DIR_SALIDA = f"/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Vision/Resultados_Test/"
MODEL_DIR = os.path.join(OUTPUT_DIR, "modelo_final")

DIR_SALIDA = OUTPUT_DIR + "/modelo_final"
os.makedirs(DIR_SALIDA, exist_ok=True)

# Archivos de salida (añadimos el nombre del modelo y los frames para no sobreescribir pruebas antiguas)
CSV_SALIDA_MAX = os.path.join(DIR_SALIDA, f"predicciones_{MODELO_ELEGIDO}_f{NUM_FOTOGRAMAS_POR_VIDEO}_max.csv")
CSV_SALIDA_MEAN = os.path.join(DIR_SALIDA, f"predicciones_{MODELO_ELEGIDO}_f{NUM_FOTOGRAMAS_POR_VIDEO}_mean.csv")
CSV_SALIDA_MAJORITY = os.path.join(DIR_SALIDA, f"predicciones_{MODELO_ELEGIDO}_f{NUM_FOTOGRAMAS_POR_VIDEO}_majority.csv")

## 2.2. Carga de datos y del modelo

In [113]:
print(f"Cargando el dataset estático de test ({NUM_FOTOGRAMAS_POR_VIDEO} frames)...")
test_df = pd.read_csv(CSV_IMAGENES_TEST)

Cargando el dataset estático de test (2 frames)...


In [114]:
print(f"Cargando procesador y modelo {MODELO_ELEGIDO.upper()} entrenado...")
processor = AutoImageProcessor.from_pretrained(MODEL_DIR)
model = AutoModelForImageClassification.from_pretrained(MODEL_DIR).to(device)
model.eval()

Cargando procesador y modelo CONVNEXT entrenado...


Loading weights:   0%|          | 0/182 [00:00<?, ?it/s]

ConvNextForImageClassification(
  (convnext): ConvNextModel(
    (embeddings): ConvNextEmbeddings(
      (patch_embeddings): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
      (layernorm): ConvNextLayerNorm((96,), eps=1e-06, elementwise_affine=True)
    )
    (encoder): ConvNextEncoder(
      (stages): ModuleList(
        (0): ConvNextStage(
          (downsampling_layer): ModuleList()
          (layers): ModuleList(
            (0-2): 3 x ConvNextLayer(
              (dwconv): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
              (layernorm): ConvNextLayerNorm((96,), eps=1e-06, elementwise_affine=True)
              (pwconv1): Linear(in_features=96, out_features=384, bias=True)
              (act): GELUActivation()
              (pwconv2): Linear(in_features=384, out_features=96, bias=True)
              (drop_path): Identity()
            )
          )
        )
        (1): ConvNextStage(
          (downsampling_layer): ModuleList(
          

## 2.3. Inferencia y recopilación de probabilidades

In [115]:
print(f"Iniciando inferencia sobre {test_df['id_EXIST'].nunique()} vídeos de test...")
predicciones_por_video = {}

for index, row in tqdm(test_df.iterrows(), total=len(test_df)):
    id_vid = row['id_EXIST']
    img_path = row['path_imagen']
    true_label = row['label']

    try:
        image = Image.open(img_path).convert("RGB")
    except Exception as e:
        # Silenciamos el print para no romper la barra tqdm
        continue

    inputs = processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]
        prob_misogino = probs[1].item()

    if id_vid not in predicciones_por_video:
        predicciones_por_video[id_vid] = {'probs': [], 'true_label': true_label}

    predicciones_por_video[id_vid]['probs'].append(prob_misogino)

Iniciando inferencia sobre 502 vídeos de test...


100%|██████████| 1004/1004 [00:21<00:00, 46.04it/s]


## 2.4. Estrategias de pooling y evaluación

Las 3 estrategias de pooling que vamos a usar son:
- *Max-pooling*: con que solamente 1 solo fotograma de los 4 sea etiquetado como misógino, el video entero se marcará como misógino.
- *Average pooling*: Se cogerá el porcentaje de seguridad del modelo para los 4 fotogramas y haremos la media. Si la media >= 50% se cataloga el vídeo como misógino.
- *Majority vote*: Hacemos que al menos 2 o 3 fotogramas sean clasificados como misóginos para considerar el vídeo como misógino.

In [116]:
y_true = []
y_pred_max, y_pred_mean, y_pred_majority = [], [], []
resultados_max, resultados_mean, resultados_majority = [], [], []

for id_vid, data in predicciones_por_video.items():
    true_label = data['true_label']
    y_true.append(true_label)

    probabilidades = data['probs']
    total_frames_analizados = len(probabilidades)
    predicciones_binarias = [1 if p > 0.5 else 0 for p in probabilidades]

    # --- 4.1 Max-Pooling ---
    max_prob = max(probabilidades) if probabilidades else 0.0
    pred_max = 1 if max_prob > 0.5 else 0
    y_pred_max.append(pred_max)
    resultados_max.append({"id_EXIST": id_vid, "prob_misogino": max_prob, "prediccion_binaria": pred_max, "label_real": true_label})

    # --- 4.2 Mean-Pooling ---
    mean_prob = sum(probabilidades) / total_frames_analizados if total_frames_analizados > 0 else 0.0
    pred_mean = 1 if mean_prob > 0.5 else 0
    y_pred_mean.append(pred_mean)
    resultados_mean.append({"id_EXIST": id_vid, "prob_misogino": mean_prob, "prediccion_binaria": pred_mean, "label_real": true_label})

    # --- 4.3 Votación por Mayoría ---
    votos_misogino = sum(predicciones_binarias)
    prob_majority = votos_misogino / total_frames_analizados if total_frames_analizados > 0 else 0.0
    pred_majority = 1 if votos_misogino >= (total_frames_analizados / 2) else 0
    y_pred_majority.append(pred_majority)
    resultados_majority.append({"id_EXIST": id_vid, "prob_misogino": prob_majority, "prediccion_binaria": pred_majority, "label_real": true_label})

# Guardamos los CSVs
pd.DataFrame(resultados_max).to_csv(CSV_SALIDA_MAX, index=False)
pd.DataFrame(resultados_mean).to_csv(CSV_SALIDA_MEAN, index=False)
pd.DataFrame(resultados_majority).to_csv(CSV_SALIDA_MAJORITY, index=False)

## 2.5. Reporte final

In [117]:
print(f"\n✅ ¡CSVs generados y guardados en {DIR_SALIDA}!")


✅ ¡CSVs generados y guardados en /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Vision/CONVNEXT_Frames_2_experimentacion/modelo_final!


In [118]:
print("\n" + "="*50)
print(f"🏆 RESULTADOS: {MODELO_ELEGIDO.upper()} ({NUM_FOTOGRAMAS_POR_VIDEO} Frames) - MAX-POOLING")
print("="*50)
print(f"F1-Score (Macro): {f1_score(y_true, y_pred_max, average='macro'):.4f}")
print(f"Accuracy:         {accuracy_score(y_true, y_pred_max):.4f}")


🏆 RESULTADOS: CONVNEXT (2 Frames) - MAX-POOLING
F1-Score (Macro): 0.4140
Accuracy:         0.4861


In [119]:
print("\n" + "-"*50)
print(f"📊 RESULTADOS: {MODELO_ELEGIDO.upper()} ({NUM_FOTOGRAMAS_POR_VIDEO} Frames) - MEAN-POOLING")
print("-"*50)
print(f"F1-Score (Macro): {f1_score(y_true, y_pred_mean, average='macro'):.4f}")
print(f"Accuracy:         {accuracy_score(y_true, y_pred_mean):.4f}")


--------------------------------------------------
📊 RESULTADOS: CONVNEXT (2 Frames) - MEAN-POOLING
--------------------------------------------------
F1-Score (Macro): 0.4689
Accuracy:         0.5040


In [120]:
print("\n" + "-"*50)
print(f"🗳️ RESULTADOS: {MODELO_ELEGIDO.upper()} ({NUM_FOTOGRAMAS_POR_VIDEO} Frames) - MAJORITY VOTE")
print("-"*50)
print(f"F1-Score (Macro): {f1_score(y_true, y_pred_majority, average='macro'):.4f}")
print(f"Accuracy:         {accuracy_score(y_true, y_pred_majority):.4f}")


--------------------------------------------------
🗳️ RESULTADOS: CONVNEXT (2 Frames) - MAJORITY VOTE
--------------------------------------------------
F1-Score (Macro): 0.4140
Accuracy:         0.4861
